# slm_prod — master pipeline (Colab, 15 GB GPU)

<a href="https://colab.research.google.com/github/Dushyantgehlot/slm_prod/blob/main/notebooks/master_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data prep → QLoRA SFT → merge → 4-bit GPTQ → benchmark all three checkpoints → comparison table, in one notebook.

This is the end-to-end equivalent of notebooks `00`–`04`, rewritten to fit a **free Colab T4 (15 GB VRAM / ~12.7 GB RAM)**. It is not a concatenation of them — four stages had to change to fit the budget:

| Stage | `00`–`04` did | Here | Why |
|---|---|---|---|
| Train | `bf16=True` | auto `bf16`/`fp16` from compute capability | T4 is sm_75 — **no bf16**, `bf16=True` raises |
| Merge | reload the 8B base at bf16, `device_map="auto"` | Unsloth `save_pretrained_merged` from the already-loaded 4-bit model | 8B at bf16 ≈ 16 GB — over VRAM *and* over Colab RAM |
| Eval | `dtype=bfloat16` (~16 GB of weights) | all three checkpoints in 4-bit | fp16 weights don't fit 15 GB |
| Eval | SFT eval needs the 16 GB merged model | SFT eval'd as base + `peft=` adapter | skips a 16 GB load and 16 GB of disk |
| Eval | one `lm_eval` call, no `--num_fewshot` | one call per shot-count group | the README's 5/25/10-shot counts were never actually applied |
| Quantize | in-process after training | subprocess | the OS is the only reliable way to reclaim training VRAM |
| Base repo | canonical `google/gemma-4-E4B` (~16 GB bf16) | auto-picks Unsloth's pre-quantized `gemma-4-E4B-unsloth-bnb-4bit` (~5 GB) when disk is tight | ~11 GB of Colab disk; switchable, since `configs/model.yaml` records a stalled download on that repo |

**Which model tier?** A free T4 is sm_75, so it has no bf16 — and Unsloth reports `Using float16 precision for gemma4 won't work! Using float32`, because Gemma's activations overflow fp16's 65504 ceiling. That fp32 fallback is a correctness requirement, not a tunable, and it roughly doubles activation memory:

| Tier | Total / active | 4-bit weights | QLoRA (bf16) | QLoRA (fp32, T4) | Fits 14.56 GB? |
|---|---|---|---|---|---|
| **E2B** | ~5.1B / ~2.3B | ~2.9 GB | 4–5 GB | ~8–10 GB | **yes** |
| E4B | ~9B / ~4B | ~4.5 GB | 8–10 GB | ~16–20 GB | no — OOMs |

So `MODEL_TIER = "auto"` picks **E2B on a T4** and E4B only where bf16 is available (L4/A100). The widely-quoted "E4B QLoRA needs 8–10 GB" assumes bf16 activations and does not hold on a T4.

**VRAM budget by stage** (T4 / E2B / `max_seq_length=1024`). Measured as the notebook runs and printed back in §6.3:

| Stage | Budget |
|---|---|
| 2 · QLoRA SFT | ~8–10 GB |
| 3 · Merge to fp16 | ~1 GB (shard-streamed to disk) |
| 4 · GPTQ quantize | ~4 GB (layer-by-layer) |
| 5 · Eval, 4-bit | ~5–7 GB |

## Before you start

1. **Runtime → Change runtime type → T4 GPU.** L4/A100 also work and are faster; the notebook detects the GPU and picks dtypes accordingly.
2. Accept the [Gemma license](https://huggingface.co/google/gemma-4-E4B) on your HF account — the weights are gated.
3. Put your token in Colab **Secrets** (🔑 in the left sidebar) as `HF_TOKEN`, notebook access on. Otherwise you'll be prompted for it.
4. Run top to bottom. There's one **restart checkpoint** after Stage 2; everything before it is written to disk, so restarting there costs nothing.

**Time budget.** `MODE = "quick"` (the default, in §0.5) caps training at 500 steps and evaluation at 40 examples per task — roughly 2.5–3.5 h on a T4, which fits one free session. `MODE = "full"` is 3 epochs plus the complete suite: **12+ hours**, more than a free session gives you in one sitting. For a full run, turn on Drive persistence in §0.4 and do it across sessions, or use a paid runtime.

## Stage 0 — Preflight, install, auth

### 0.1 · What GPU did we get?

Records the two facts every later stage branches on: how much VRAM there is, and whether the card can do bf16.

In [ ]:
import os, shutil, subprocess, sys

# MUST be set before torch creates a CUDA context — importing torch and touching
# torch.cuda below is what initialises the allocator, and it reads this once at
# init. Setting it later (as an earlier draft of this notebook did in §0.4) is a
# no-op, which is exactly the fragmentation the OOM message warns about.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

IN_COLAB = "google.colab" in sys.modules

assert torch.cuda.is_available(), (
    "No GPU attached. Runtime > Change runtime type > T4 GPU, then re-run this cell."
)

props = torch.cuda.get_device_properties(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = props.total_memory / 1024**3
# bf16 needs Ampere (sm_80+). torch.cuda.is_bf16_supported() reports True on sm_75
# via slow emulation, so gate on the capability directly instead.
BF16_OK = CAPABILITY[0] >= 8
COMPUTE_DTYPE = "bfloat16" if BF16_OK else "float16"

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1024**3
except ImportError:
    RAM_GB = float("nan")
DISK_FREE_GB = shutil.disk_usage("/content" if IN_COLAB else ".").free / 1024**3

print(f"GPU          : {props.name} (sm_{CAPABILITY[0]}{CAPABILITY[1]})")
print(f"VRAM         : {VRAM_GB:.1f} GB")
print(f"System RAM   : {RAM_GB:.1f} GB")
print(f"Free disk    : {DISK_FREE_GB:.1f} GB")
print(f"bf16 support : {BF16_OK}  ->  compute dtype = {COMPUTE_DTYPE}")

if VRAM_GB < 14:
    print(f"\n[WARN] {VRAM_GB:.1f} GB VRAM is under the 15 GB this notebook targets. "
          "Set MAX_SEQ_LENGTH = 1024 in the run-plan cell.")
# ~16 GB base download + ~16 GB merged fp16 + ~6 GB GPTQ + eval cache.
if DISK_FREE_GB < 45:
    print(f"\n[NOTE] {DISK_FREE_GB:.1f} GB free disk; the pipeline writes ~40 GB with the "
          "canonical base repo (16 GB base cache + 16 GB merged fp16 + 6 GB GPTQ). "
          "The run-plan cell will switch to Unsloth's pre-quantized 4-bit base "
          "(~5 GB instead of 16 GB) automatically; also consider "
          "FREE_MERGED_AFTER_QUANT = True in Stage 4.")

### 0.2 · Clone the repo

`src/slm_prod/` holds the config loading and quantization logic this notebook imports, so nothing important lives only in a notebook cell. Re-runnable — skips the clone if the repo is already present.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Dushyantgehlot/slm_prod.git"
WORKDIR = Path("/content") if IN_COLAB else Path.cwd().parent

if (Path.cwd() / "src" / "slm_prod").exists():
    REPO = Path.cwd()                       # running from inside a local checkout
elif (WORKDIR / "slm_prod" / "src").exists():
    REPO = WORKDIR / "slm_prod"             # already cloned earlier this session
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR / "slm_prod")], check=True
    )
    REPO = WORKDIR / "slm_prod"

os.chdir(REPO)
print(f"Repo root: {REPO}")

### 0.3 · Install dependencies

Colab's GPU image ships a CUDA-matched torch/transformers/accelerate — `requirements-colab.txt` is the trimmed set that deliberately does *not* reinstall them. Two additions on top of it:

- `lm-eval[ifeval]` pulls the `langdetect` / `immutabledict` / `nltk` extras IFEval needs and plain `lm-eval` omits.
- `tabulate` backs `DataFrame.to_markdown()` in §6.3.

⚠️ **This cell often forces a restart** — installing Unsloth moves the `numpy`/`torch` pins. Read the banner it prints.

In [ ]:
%pip install -q -r requirements-colab.txt
%pip install -q -e .
%pip install -q "lm-eval[ifeval]>=0.4.4" tabulate

print("\n" + "=" * 72)
print("If pip reported dependency conflicts or a numpy/torch change above:")
print("  Runtime > Restart session, then re-run 0.1, 0.2 and 0.4 — NOT this cell.")
print("=" * 72)

### 0.4 · Environment, HF auth, optional Drive persistence

`expandable_segments` is the highest-value VRAM knob here: training and eval both allocate variable-length sequences, and fragmentation alone can cost a gigabyte on a 15 GB card.

Drive is optional but worth it for `MODE = "full"` — free sessions disconnect, and the adapter plus results are small enough to keep there.

In [ ]:
import os
from pathlib import Path

# PYTORCH_CUDA_ALLOC_CONF is deliberately NOT set here — it has to precede the
# first torch CUDA call, so it lives at the top of §0.1.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault(
    "HF_HOME",
    str(Path("/content/hf_cache") if IN_COLAB else Path.home() / ".cache" / "huggingface"),
)

# --- HF auth (Gemma weights are gated) --------------------------------------
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token and IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    from getpass import getpass
    hf_token = getpass("HF_TOKEN (needs the Gemma license accepted): ")

login(token=hf_token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = hf_token
print("Hugging Face: authenticated")

# --- optional: survive a session disconnect ---------------------------------
USE_DRIVE = False

# --- fail fast on the peft/gptqmodel/optimum interaction --------------------
# PEFT calls is_gptqmodel_available() inside get_peft_model(). Having gptqmodel
# installed (which we need for Stage 4) makes that probe hard-raise unless
# optimum >= 1.24.0 is present. Catch it here rather than after a 20-minute
# model download, in the middle of Stage 2.
try:
    from peft.import_utils import is_gptqmodel_available
    is_gptqmodel_available()
    print("peft / gptqmodel / optimum: compatible")
except ImportError as exc:
    print(f"[BLOCKER] {exc}\n           run:  %pip install -q -U 'optimum>=1.24.0'  then restart")
except Exception:
    pass   # older PEFT without this probe — nothing to check

DRIVE_DIR = None
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/slm_prod")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Persisting artifacts to {DRIVE_DIR}")

### 0.5 · Run plan

**The one cell you edit.** Everything downstream reads from it.

`quick` exists because the honest full run doesn't fit a free session: 9.5 k examples × 3 epochs at effective batch 8 is ~3,560 optimizer steps, and MMLU alone is ~14 k questions × 3 checkpoints. `quick` shrinks *volume*, never *method* — same tasks, same shot counts, same 4-bit precision on all three checkpoints, so the deltas stay internally comparable. They just carry more sampling noise, and they are not comparable to published leaderboard numbers.

In [ ]:
import json
from pathlib import Path

from slm_prod.utils import load_config

MODE = "quick"          # "quick" -> fits one free session | "full" -> the real run
assert MODE in {"quick", "full"}

model_cfg = load_config("model.yaml")
lora_cfg  = load_config("lora.yaml")
sft_cfg   = load_config("sft.yaml")
gptq_cfg  = load_config("gptq.yaml")
eval_cfg  = load_config("eval_tasks.yaml")

# --- which model tier? ------------------------------------------------------
# E4B does not fit a T4 under QLoRA, despite the documented 8-10 GB figure. That
# figure assumes bf16 activations. A T4 is sm_75, so Unsloth prints
#   "Using float16 precision for gemma4 won't work! Using float32."
# and falls back to fp32 activations -- float16 tops out at 65504 and Gemma
# overflows it, so this is a correctness fallback, not a tunable. fp32 roughly
# doubles activation memory, pushing E4B to ~16-20 GB against a 14.56 GB card.
#
#   tier   total / active   4-bit weights   QLoRA (bf16)   QLoRA (fp32, T4)
#   E2B    ~5.1B / ~2.3B    ~2.9 GB         4-5 GB         ~8-10 GB   fits
#   E4B    ~9B   / ~4B      ~4.5 GB         8-10 GB        ~16-20 GB  OOM
MODEL_TIER = "auto"          # "auto" | "E2B" | "E4B"

MODEL_REPOS = {
    "E2B": {"base": "google/gemma-4-E2B", "it": "google/gemma-4-E2B-it",
            "bnb4": "unsloth/gemma-4-E2B-unsloth-bnb-4bit"},
    "E4B": {"base": "google/gemma-4-E4B", "it": "google/gemma-4-E4B-it",
            "bnb4": "unsloth/gemma-4-E4B-unsloth-bnb-4bit"},
}

if MODEL_TIER == "auto":
    MODEL_TIER = "E4B" if (BF16_OK and VRAM_GB >= 16) else "E2B"
    print(f"tier auto-selected: {MODEL_TIER} "
          f"(bf16={BF16_OK}, {VRAM_GB:.1f} GB VRAM)")
assert MODEL_TIER in MODEL_REPOS

MAX_SEQ_LENGTH = model_cfg["max_seq_length"]
# fp32 activations scale with sequence length, so shorten the window on a card
# that can't do bf16. Override explicitly if you'd rather trade batch for length.
if not BF16_OK:
    MAX_SEQ_LENGTH = min(MAX_SEQ_LENGTH, 1024)
    TRAIN_BATCH_SIZE, GRAD_ACCUM = 1, 8      # same effective batch of 8
else:
    TRAIN_BATCH_SIZE = sft_cfg["per_device_train_batch_size"]
    GRAD_ACCUM = sft_cfg["gradient_accumulation_steps"]

# --- which base repo? -------------------------------------------------------
# The canonical google/ repo downloads ~16 GB of bf16 weights that bitsandbytes
# then quantizes on load. Unsloth's pre-quantized mirror is ~5 GB and skips that
# step, which is ~11 GB of disk and several minutes of Colab network back.
# It is NOT free of risk: configs/model.yaml records a hard DownloadStallError
# on that repo (low-traffic, Xet + HTTP fallback both exhausted). So: prefer it
# when disk is tight, and fall back to the canonical repo if it stalls.
# Note: Unsloth is NOT restricted to unsloth/* repos — it loads any transformers
# model. But FastLanguageModel.from_pretrained resolves known names through its
# own registry, so passing google/gemma-4-E2B with load_in_4bit=True makes it
# fetch unsloth/gemma-4-E2B-unsloth-bnb-4bit anyway (you'll see the substitution
# in its LOAD REPORT). Those uploads also carry Gemma 4 fixes -- notably a
# gradient-accumulation bug that sent losses to 300-400 instead of 10-15, which
# matters here because this config uses grad accum. Needs unsloth >= 2026.4.4.
# This setting therefore mostly affects the eval/bench stages, which go through
# plain transformers and get no such redirect.
PREFER_UNSLOTH_4BIT = "auto"     # "auto" (by free disk) | True (always) | False (never)

if PREFER_UNSLOTH_4BIT == "auto":
    use_unsloth_repo = DISK_FREE_GB < 45
else:
    use_unsloth_repo = bool(PREFER_UNSLOTH_4BIT)

repos = MODEL_REPOS[MODEL_TIER]
BASE_MODEL = repos["bnb4"] if use_unsloth_repo else repos["base"]
IT_MODEL   = repos["it"]          # chat-template source; see Stage 1
# A pre-quantized bnb repo carries its own quantization_config; re-passing
# load_in_4bit downstream would fight with it, so later stages branch on this.
BASE_IS_PREQUANTIZED = use_unsloth_repo

print(f"base model: {BASE_MODEL}"
      + (f"  (pre-quantized 4-bit; {DISK_FREE_GB:.0f} GB free disk)" if use_unsloth_repo
         else "  (bf16 weights, quantized on load)"))

# --- training volume --------------------------------------------------------
TRAIN_MAX_STEPS = 500 if MODE == "quick" else -1    # -1 -> fall back to num_train_epochs
TRAIN_SUBSAMPLE = None                              # e.g. 2000 to cap training rows
EVAL_SUBSAMPLE  = 200                               # in-training loss only; keep small

# --- benchmark volume -------------------------------------------------------
EVAL_LIMIT      = 40 if MODE == "quick" else None   # examples per task (per MMLU subtask!)
EVAL_BATCH_SIZE = "auto"                            # set to 1 if auto overshoots into OOM

# Canonical shot counts from the README. run_eval.sh passes no --num_fewshot, so
# those counts were aspirational; here each group is its own lm_eval call.
FEWSHOT = {
    "mmlu": 5,
    "arc_challenge": 25,
    "hellaswag": 10,
    "gsm8k": 5,
    "truthfulqa_mc2": 0,
    "ifeval": 0,
}
TASKS = [t for t in eval_cfg["tasks"] if t in FEWSHOT]

# --- paths ------------------------------------------------------------------
# The config paths hardcode "e4b"; rewrite to the tier actually in use so an E2B
# run neither mislabels its artifacts nor overwrites an E4B one.
def tier_path(configured: str) -> Path:
    return REPO / configured.replace("e4b", MODEL_TIER.lower())

ADAPTER_DIR = tier_path(sft_cfg["adapter_dir"])
MERGED_DIR  = tier_path(sft_cfg["merged_dir"])
GPTQ_DIR    = tier_path(gptq_cfg["output_dir"])
OUTPUT_DIR  = tier_path(sft_cfg["output_dir"])
RESULTS_DIR = REPO / "eval" / "results"
RUN_DIR     = REPO / ".master_run"                  # generated stage scripts + job files
for d in (ADAPTER_DIR.parent, MERGED_DIR.parent, RESULTS_DIR, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"\ntier={MODEL_TIER}  seq_len={MAX_SEQ_LENGTH}  batch={TRAIN_BATCH_SIZE}x{GRAD_ACCUM}"
      f"  activations={'bf16' if BF16_OK else 'fp32 (T4 fallback)'}")
print(f"MODE={MODE}  max_steps={TRAIN_MAX_STEPS}  eval_limit={EVAL_LIMIT}")
print("tasks: " + ", ".join(f"{t} ({FEWSHOT[t]}-shot)" for t in TASKS))
if EVAL_LIMIT and "mmlu" in TASKS:
    print(f"\nnote: --limit is per subtask, so mmlu alone is 57 x {EVAL_LIMIT} "
          f"= {57 * EVAL_LIMIT} items per checkpoint.")

VRAM bookkeeping. Every stage calls this so §6.3 can report measured peaks instead of the estimates in the table at the top.

In [ ]:
import gc

VRAM_LOG = {}

def vram_reset():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def vram_report(stage: str) -> float:
    """Peak vs resident matters more than it looks. Loading spikes transiently
    (dequant buffers, shard staging) and that memory is handed back; what limits
    the NEXT stage is what stays resident. Reporting only the peak makes a run
    look far tighter than it is."""
    peak = torch.cuda.max_memory_allocated() / 1024**3
    torch.cuda.empty_cache()
    live = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    VRAM_LOG[stage] = peak
    print(f"[vram] {stage}\n"
          f"        peak {peak:5.2f} | resident {live:5.2f} | reserved {reserved:5.2f} GB"
          f"  ->  {VRAM_GB - live:5.2f} GB free for the next stage")
    return live

def free_vram(*names):
    for n in names:
        globals().pop(n, None)
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[vram] after teardown: allocated {torch.cuda.memory_allocated()/1024**3:.2f} GB, "
          f"reserved {torch.cuda.memory_reserved()/1024**3:.2f} GB")

def run_logged(cmd: list[str], log_name: str) -> int:
    """Run a subprocess, mirroring its output into the cell and tee-ing to a file.

    subprocess.run() inherits the kernel's stdout, which Colab does NOT reliably
    forward into cell output — so a failing stage prints "see the traceback above"
    with nothing above it. Piping and echoing keeps the traceback where you can
    actually read it, and the log file survives the cell being cleared."""
    log_path = RUN_DIR / f"{log_name}.log"
    lines: list[str] = []
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    proc.wait()
    log_path.write_text("".join(lines))
    if proc.returncode != 0:
        tail = "".join(lines[-30:]) or "(the subprocess produced no output at all)"
        print(f"\n{'-' * 72}\nexit {proc.returncode}. Last lines:\n{tail}"
              f"\nFull log: {log_path}\n{'-' * 72}")
    return proc.returncode

vram_reset()

## Stage 1 — Data prep

`HuggingFaceH4/no_robots`: ~9.5 k human-written instruction pairs, CC-BY-4.0, with a held-out test split. Chosen over `FineTome-100k` (the dataset in the Unsloth Gemma 4 recipe this config was benchmarked against) so these numbers are an independent measurement rather than a reproduction — see [`data/README.md`](../data/README.md).

Three things happen beyond what `src/slm_prod/data.py` does on its own:

- **The chat template is resolved explicitly, and reused.** Two separate problems stack here. First, `google/gemma-4-E4B` is a *base* repo, and base repos ship no chat template — only `-it` mirrors do. Second, and more surprising: **Gemma 4 stores its template in a standalone `chat_template.jinja`, not in `tokenizer_config.json`**, and `AutoTokenizer` only reads the latter — so `tokenizer.chat_template` is `None` even for `gemma-4-E4B-it` ([transformers#45205](https://github.com/huggingface/transformers/issues/45205), [E2B-it discussion](https://huggingface.co/google/gemma-4-E2B-it/discussions/8)). The cell below therefore pulls the `.jinja` directly via `hf_hub_download`, falling back through the base repo → the `-it` mirror → Unsloth's built-ins, and renders one message pair to prove the template executes before 9.5 k rows go through it.

  The resolved template is then pinned to disk and reused verbatim in Stage 2. The trainer only ever sees the pre-rendered `text` column, so a mismatch between prep and generation wouldn't raise — it would quietly produce a model that never emits its own turn separators.
- **System turns get folded into the first user turn.** Gemma chat templates have historically rejected a standalone `system` role, and `no_robots` has system turns. Rather than assume which way Gemma 4's template goes, the cell below probes it once and only rewrites if the template actually rejects it.
- **Over-length rows are dropped, not truncated.** A truncated assistant turn teaches the model to stop mid-sentence. At 2048 tokens this costs very few rows — the cell prints how many.

In [ ]:
from unsloth import FastLanguageModel   # import before transformers/trl so the patches land
from transformers import AutoTokenizer

from huggingface_hub import hf_hub_download

tokenizer_probe = AutoTokenizer.from_pretrained(BASE_MODEL)

def fetch_jinja(repo_id: str) -> str | None:
    """Gemma 4 ships its chat template as a standalone chat_template.jinja, which
    AutoTokenizer does not read (huggingface/transformers#45205). Fetch it directly."""
    try:
        return Path(hf_hub_download(repo_id, "chat_template.jinja")).read_text()
    except Exception as exc:
        print(f"  no chat_template.jinja in {repo_id} ({type(exc).__name__})")
        return None

CHAT_TEMPLATE = tokenizer_probe.chat_template
CHAT_TEMPLATE_SOURCE = f"{BASE_MODEL} tokenizer_config.json"

if not CHAT_TEMPLATE:
    print("tokenizer_config.json carries no chat_template — expected for Gemma 4 "
          "(template lives in a separate .jinja) and for base repos generally.")
    # dict.fromkeys dedupes while preserving order, in case BASE_MODEL is already the -it repo.
    for repo in dict.fromkeys([BASE_MODEL, IT_MODEL]):
        CHAT_TEMPLATE = fetch_jinja(repo)
        if CHAT_TEMPLATE:
            CHAT_TEMPLATE_SOURCE = f"{repo}/chat_template.jinja"
            break

if not CHAT_TEMPLATE and model_cfg.get("chat_template"):
    try:
        from unsloth.chat_templates import get_chat_template
        tokenizer_probe = get_chat_template(tokenizer_probe, chat_template=model_cfg["chat_template"])
        CHAT_TEMPLATE = tokenizer_probe.chat_template
        CHAT_TEMPLATE_SOURCE = f"unsloth '{model_cfg['chat_template']}'"
    except Exception as exc:
        print(f"  unsloth '{model_cfg['chat_template']}' unavailable ({type(exc).__name__}) — "
              "unsloth's keys are gemma / gemma2 / gemma-3; 'gemma4' is not one of them")

assert CHAT_TEMPLATE, (
    f"no chat template found: not in {BASE_MODEL}'s tokenizer_config.json, not as a "
    f"chat_template.jinja in {BASE_MODEL} or {IT_MODEL}, and not available from "
    f"unsloth. Check {IT_MODEL}'s license is accepted on your HF account."
)

tokenizer_probe.chat_template = CHAT_TEMPLATE
# Render once to prove the template actually executes before 9.5k rows go through it.
_probe = tokenizer_probe.apply_chat_template(
    [{"role": "user", "content": "ping"}, {"role": "assistant", "content": "pong"}],
    tokenize=False,
)
(RUN_DIR / "chat_template.jinja").write_text(CHAT_TEMPLATE)
print(f"chat template source: {CHAT_TEMPLATE_SOURCE}  ({len(CHAT_TEMPLATE)} chars)")
print(f"renders as: {_probe!r}")

try:
    tokenizer_probe.apply_chat_template(
        [{"role": "system", "content": "s"}, {"role": "user", "content": "u"}], tokenize=False
    )
    SYSTEM_ROLE_OK = True
except Exception as exc:
    SYSTEM_ROLE_OK = False
    print(f"template rejects a standalone system turn ({type(exc).__name__}) "
          "-> folding system into the first user turn")

print(f"system role supported: {SYSTEM_ROLE_OK}")

In [ ]:
from datasets import load_dataset

def normalise_messages(messages: list[dict]) -> list[dict]:
    """Drop empty turns; fold a leading system turn into the first user turn when
    the chat template cannot represent one."""
    msgs = [m for m in messages if (m.get("content") or "").strip()]
    if SYSTEM_ROLE_OK or not msgs or msgs[0]["role"] != "system":
        return msgs
    system, rest = msgs[0], msgs[1:]
    if rest and rest[0]["role"] == "user":
        rest[0] = {"role": "user", "content": f"{system['content']}\n\n{rest[0]['content']}"}
        return rest
    return [{"role": "user", "content": system["content"]}] + rest

def build_split(split: str, subsample: int | None = None):
    ds = load_dataset(sft_cfg["dataset_id"], split=split)
    if subsample:
        ds = ds.shuffle(seed=sft_cfg["seed"]).select(range(min(subsample, len(ds))))

    def _render(ex):
        text = tokenizer_probe.apply_chat_template(
            normalise_messages(ex["messages"]), tokenize=False, add_generation_prompt=False
        )
        return {"text": text, "n_tokens": len(tokenizer_probe(text).input_ids)}

    ds = ds.map(_render, remove_columns=ds.column_names, desc=f"render {split}")
    before = len(ds)
    ds = ds.filter(lambda ex: 0 < ex["n_tokens"] <= MAX_SEQ_LENGTH, desc=f"filter {split}")
    print(f"{split}: {before} rows -> {len(ds)} "
          f"(dropped {before - len(ds)} empty or over {MAX_SEQ_LENGTH} tokens)")
    return ds

train_dataset = build_split(sft_cfg["dataset_split"], TRAIN_SUBSAMPLE)
eval_dataset  = build_split(sft_cfg["eval_split"], EVAL_SUBSAMPLE)

In [ ]:
import numpy as np

lengths = np.array(train_dataset["n_tokens"])
print(f"train rows        : {len(train_dataset)}")
print(f"tokens/row  mean  : {lengths.mean():.0f}")
print(f"            p50   : {np.percentile(lengths, 50):.0f}")
print(f"            p95   : {np.percentile(lengths, 95):.0f}")
print(f"            max   : {lengths.max()}")
print(f"total train tokens: {lengths.sum():,}")

eff_batch = TRAIN_BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = max(len(train_dataset) // eff_batch, 1)
planned = TRAIN_MAX_STEPS if TRAIN_MAX_STEPS > 0 else steps_per_epoch * sft_cfg["num_train_epochs"]
print(f"\neffective batch   : {eff_batch}")
print(f"steps/epoch       : {steps_per_epoch}")
print(f"planned steps     : {planned}  ({planned / steps_per_epoch:.2f} epochs)")

print("\n--- sample rendered example " + "-" * 45)
print(train_dataset[0]["text"][:800])

In [ ]:
# Keep the formatted splits so a restart doesn't re-download and re-render them.
PREPPED_DIR = REPO / "data" / "prepared"
train_dataset.save_to_disk(str(PREPPED_DIR / "train"))
eval_dataset.save_to_disk(str(PREPPED_DIR / "eval"))
print(f"prepared splits -> {PREPPED_DIR}")

del tokenizer_probe
gc.collect()

## Stage 2 — QLoRA SFT

Where the 15 GB actually gets spent, for **E2B on a T4** (the auto-selected tier — see §0.5 for why E4B doesn't fit):

| Item | Cost |
|---|---|
| ~5.1B weights, NF4 | ~2.9 GB |
| LoRA adapters (r=16, text tower only) | ~30 MB |
| `adamw_8bit` states (adapters only) | ~60 MB |
| Activations, batch 1 × 1024, **fp32**, Unsloth checkpointing | ~4–6 GB |
| Fragmentation + cuBLAS workspace | ~1 GB |

The activation row is the one that decides everything. It's fp32 rather than bf16 because a T4 can't do bf16 and Gemma overflows fp16 — and it's the term that scales with sequence length, which is why §0.5 halves `max_seq_length` to 1024 on a non-bf16 card.

`text_only: true` matters more than it looks. E4B is multimodal, and on a T4 Unsloth upcasts non-quantized modules to fp32; leaving the vision/audio towers loaded adds enough pressure that `device_map` starts offloading to CPU — which bitsandbytes 4-bit rejects outright with *"Some modules are dispatched on the CPU or the disk"*.

In [ ]:
import inspect

vram_reset()

load_kwargs = dict(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,                                  # let Unsloth pick for the detected GPU
    load_in_4bit=model_cfg["load_in_4bit"],
)
# text_only is a newer Unsloth kwarg — pass it only if this build accepts it.
if "text_only" in inspect.signature(FastLanguageModel.from_pretrained).parameters:
    load_kwargs["text_only"] = model_cfg["text_only"]
else:
    print("[warn] this Unsloth build has no text_only kwarg; the vision/audio towers will "
          "load and VRAM will be tighter. Upgrade Unsloth if you hit a CPU-offload error.")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
except torch.cuda.OutOfMemoryError:
    raise RuntimeError(
        f"OOM loading {BASE_MODEL} ({MODEL_TIER}) into {VRAM_GB:.1f} GB.\n"
        f"  activations: {'bf16' if BF16_OK else 'fp32 — Unsloth falls back on sm_75, ~2x memory'}\n"
        "  fixes, in order: set MODEL_TIER = 'E2B' in §0.5 (E4B needs ~16-20 GB under\n"
        "  fp32); lower MAX_SEQ_LENGTH; or switch to an L4/A100 runtime, which has bf16\n"
        "  and avoids the fp32 fallback entirely."
    ) from None

# text_only=True builds the text tower without going through the multimodal config,
# which leaves config.architectures as None. Two things break on that:
#   1. Unsloth's generate() sniffs for a VLM with
#      `any(x.endswith(...) for x in self.config.architectures)` -> TypeError.
#   2. It is written into config.json on save, and a checkpoint with no
#      `architectures` breaks AutoModel / GPTQModel reloads in Stages 4-6.
# Setting it to the concrete class is correct here: text-only means not a VLM.
if getattr(model.config, "architectures", None) is None:
    model.config.architectures = [type(model).__name__]
    print(f"config.architectures was unset (text_only side effect) -> {model.config.architectures}")

vram_report("2 · base model loaded (NF4)")

In [ ]:
# Reuse the exact template Stage 1 rendered the dataset with. Resolving it a second
# time here would risk training on one format and generating with another — the
# training tokenizer only ever sees the pre-rendered `text` column, so a mismatch
# would surface silently as a model that never emits its own turn separators.
CHAT_TEMPLATE = globals().get("CHAT_TEMPLATE") or (RUN_DIR / "chat_template.jinja").read_text()
tokenizer.chat_template = CHAT_TEMPLATE
assert tokenizer.chat_template
print(f"chat template: {len(CHAT_TEMPLATE)} chars, identical to the one Stage 1 rendered with")

### 2.1 · Qualitative baseline

Generate from the **un-tuned** model first. Benchmark deltas tell you whether something moved; this tells you what it looks like.

In [ ]:
BASELINE_PROMPT = "Explain what a GPTQ quantized model is, in two sentences, to a non-technical reader."

def chat_generate(model, tokenizer, prompt: str, max_new_tokens: int = 120) -> str:
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

FastLanguageModel.for_inference(model)
baseline_generation = chat_generate(model, tokenizer, BASELINE_PROMPT)
print(baseline_generation)

### 2.2 · Attach LoRA adapters

`target_modules` has to be a **full-path regex**, not a leaf-name list. The vision/audio towers reuse the same leaf names (`q_proj`, `k_proj`, …) but wrap them in a `Gemma4ClippableLinear` that PEFT can't adapt — a plain `["q_proj", ...]` list matches those too, and PEFT's type check rejects the match before any `exclude_modules` rule gets to run. Needs `peft>=0.19`.

But the *right* regex depends on how the model was loaded, which is why `configs/lora.yaml`'s hardcoded one fails here:

| Load | Module path | Prefix |
|---|---|---|
| `text_only=False` (multimodal wrapper) | `…language_model.layers.0.self_attn.q_proj` | `language_model.` |
| `text_only=True` (`Gemma4ForCausalLM`) | `model.layers.0.self_attn.q_proj` | none |

Since §2 loads with `text_only=True`, the config's `.*language_model\.layers\.…` matches **nothing**, and PEFT raises `Target modules … not found in the base model`. So the cell below derives the prefix from the live module tree instead of assuming either layout, then asserts the result matches something and stays out of the towers. A regex that silently matches nothing is the dangerous case: training runs, loss moves, and no weights that matter are being updated.

In [ ]:
import re

LEAF_PROJECTIONS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
TOWERS = ("vision_tower", "audio_tower")

all_modules = [n for n, _ in model.named_modules()]
text_projections = [
    n for n in all_modules
    if n.split(".")[-1] in LEAF_PROJECTIONS and not any(t in n for t in TOWERS)
]
assert text_projections, "found no text-tower projections — has the module layout changed?"

# Derive the prefix that precedes `layers.<idx>.` instead of hardcoding one. Under
# text_only=True the model is a Gemma4ForCausalLM ("model.layers.0.self_attn.q_proj");
# the multimodal wrapper nests it one level deeper ("...language_model.layers.0...").
# configs/lora.yaml hardcodes the latter, so it matches nothing here.
prefix = re.match(r"^(.*?)layers\.\d+\.", text_projections[0]).group(1)
TARGET_REGEX = (rf"^{re.escape(prefix)}layers\.\d+\."
                r"(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$")

matched = [n for n in all_modules if re.match(TARGET_REGEX, n)]
assert matched, f"derived regex matched nothing: {TARGET_REGEX}"
assert not [n for n in matched if any(t in n for t in TOWERS)], "regex leaked into a tower"

print(f"module prefix : {prefix!r}")
print(f"target regex  : {TARGET_REGEX}")
print(f"matches       : {len(matched)} projections "
      f"(of {len(text_projections)} text-tower candidates)")
if lora_cfg["target_modules"] != TARGET_REGEX:
    print(f"\n[note] configs/lora.yaml's regex differs and is not used here:"
          f"\n       {lora_cfg['target_modules']}"
          f"\n       It assumes the multimodal wrapper's language_model.* nesting, which"
          f"\n       text_only=True removes. src/slm_prod/train_sft.py still uses it.")

In [ ]:
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=TARGET_REGEX,
    bias=lora_cfg["bias"],
    use_gradient_checkpointing=lora_cfg["use_gradient_checkpointing"],
    random_state=lora_cfg["random_state"],
)
model.print_trainable_parameters()

adapted = [n for n, _ in model.named_modules() if n.endswith("lora_A.default")]
assert adapted, "LoRA matched nothing"
assert not [n for n in adapted if any(t in n for t in TOWERS)], "LoRA leaked into a tower"
print(f"\n{len(adapted)} adapted projections, none in vision/audio towers")
vram_report("2 · LoRA attached")

### 2.3 · Train

Two compatibility shims, both deliberate:

- **`bf16` vs `fp16`**, chosen from the detected compute capability. `src/slm_prod/train_sft.py:69` hardcodes `bf16=True`, which raises on a T4.
- **TRL argument names.** TRL renamed `max_seq_length` → `max_length` and `tokenizer=` → `processing_class=`. The shim inspects the installed signature so this notebook works either side of that rename.

In [ ]:
from trl import SFTConfig, SFTTrainer

def build_sft_config(**kw) -> SFTConfig:
    valid = set(inspect.signature(SFTConfig.__init__).parameters)
    if "max_seq_length" in kw and "max_seq_length" not in valid:
        kw["max_length"] = kw.pop("max_seq_length")          # TRL >= 0.13 rename
    dropped = sorted(k for k in kw if k not in valid)
    if dropped:
        print(f"[compat] this TRL build ignores: {dropped}")
    return SFTConfig(**{k: v for k, v in kw.items() if k in valid})

training_args = build_sft_config(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=sft_cfg["num_train_epochs"],
    max_steps=TRAIN_MAX_STEPS,
    learning_rate=sft_cfg["learning_rate"],
    lr_scheduler_type=sft_cfg["lr_scheduler_type"],
    warmup_ratio=sft_cfg["warmup_ratio"],
    weight_decay=sft_cfg["weight_decay"],
    optim=sft_cfg["optim"],
    logging_steps=sft_cfg["logging_steps"],
    eval_strategy="steps",
    eval_steps=sft_cfg["eval_steps"],
    save_steps=sft_cfg["save_steps"],
    save_total_limit=sft_cfg["save_total_limit"],
    seed=sft_cfg["seed"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    bf16=BF16_OK,            # <- T4 fix; train_sft.py hardcodes bf16=True
    fp16=not BF16_OK,
    report_to="none",
)

tok_kwarg = ("processing_class"
             if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters
             else "tokenizer")

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset.remove_columns("n_tokens"),
    eval_dataset=eval_dataset.remove_columns("n_tokens"),
    args=training_args,
    **{tok_kwarg: tokenizer},
)
print(f"[compat] SFTTrainer tokenizer kwarg: {tok_kwarg}=")

In [ ]:
vram_reset()
trainer_stats = trainer.train()
vram_report("2 · training")

print(f"\ntrain runtime : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")
print(f"final loss    : {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

history   = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts  = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(9, 4))
if train_pts:
    ax.plot(*zip(*train_pts), label="train loss", linewidth=1.2)
if eval_pts:
    ax.plot(*zip(*eval_pts), label="eval loss (no_robots test)", marker="o", linewidth=1.2)
ax.set_xlabel("step"); ax.set_ylabel("loss")
ax.set_title(f"QLoRA SFT — gemma-4-E4B on no_robots ({MODE} mode)")
ax.legend(frameon=False); ax.grid(alpha=0.3)
plt.tight_layout()
(REPO / "reports" / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(REPO / "reports" / "figures" / "sft_loss.png", dpi=150)
plt.show()

### 2.4 · Same prompt, after SFT

In [ ]:
FastLanguageModel.for_inference(model)
sft_generation = chat_generate(model, tokenizer, BASELINE_PROMPT)

print("=== BASE ===\n" + baseline_generation)
print("\n=== SFT ===\n" + sft_generation)

### 2.5 · Save the adapter, and merge to fp16 **while the model is still loaded**

This is the stage that decides whether the pipeline fits 15 GB.

The obvious merge — what `src/slm_prod/merge_lora.py:20` does — reloads the base at bf16 with `device_map="auto"` and calls `merge_and_unload()`. For an 8B model that's ~16 GB of weights: over a T4's 15 GB *and* over Colab's 12.7 GB of RAM, so it fails on both paths (usually as a silent OOM-kill, no traceback).

`save_pretrained_merged(..., save_method="merged_16bit")` skips the reload entirely. The trained model is already resident in NF4; Unsloth dequantizes and folds the adapter **shard by shard, streaming each to disk**, so peak memory is one shard rather than one model.

One thing to be clear-eyed about: the fp16 output is a *dequantized NF4 base* plus the adapter, not the original bf16 weights. That's true whichever base repo you picked — under QLoRA the base is NF4 in memory either way — so it doesn't change the base↔SFT comparison. It does mean the GPTQ input has already been through one round of 4-bit, which is the normal QLoRA→GPTQ path but worth stating rather than glossing.

In [ ]:
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1024**2
print(f"adapter -> {ADAPTER_DIR}  ({adapter_mb:.0f} MB)")

if DRIVE_DIR:
    shutil.copytree(ADAPTER_DIR, DRIVE_DIR / ADAPTER_DIR.name, dirs_exist_ok=True)
    print(f"mirrored -> {DRIVE_DIR / ADAPTER_DIR.name}")

In [ ]:
free_gb = shutil.disk_usage(REPO).free / 1024**3
assert free_gb > 20, f"only {free_gb:.1f} GB free; the fp16 merge needs ~16 GB. Clear space first."

vram_reset()
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
vram_report("3 · merge to fp16 (shard-streamed)")

merged_gb = sum(f.stat().st_size for f in MERGED_DIR.rglob("*") if f.is_file()) / 1024**3
print(f"merged fp16 -> {MERGED_DIR}  ({merged_gb:.1f} GB)")

In [ ]:
free_vram("trainer", "model", "tokenizer", "train_dataset", "eval_dataset")

print(f"""
{'=' * 72}
RESTART CHECKPOINT — everything so far is on disk:
  adapter : {ADAPTER_DIR}
  merged  : {MERGED_DIR}

The remaining stages run as subprocesses, so they get a clean CUDA context
either way. But on a 15 GB card the safest move is:

  Runtime > Restart session, then run 0.1, 0.2, 0.4, 0.5, the VRAM-helper cell,
  and 'Resume' below. Skip 0.3, Stage 1 and Stage 2.
{'=' * 72}
""")

### Resume after a restart

Run this instead of Stages 1–2 if you restarted, or if you're picking the notebook back up in a later session with Drive persistence on. It only verifies that the artifacts Stages 1–2 produce are actually on disk.

In [ ]:
# Needs 0.1, 0.2, 0.4, 0.5 and the VRAM-helper cell to have run first.
if DRIVE_DIR and not ADAPTER_DIR.exists() and (DRIVE_DIR / ADAPTER_DIR.name).exists():
    shutil.copytree(DRIVE_DIR / ADAPTER_DIR.name, ADAPTER_DIR, dirs_exist_ok=True)
    print(f"restored adapter from Drive -> {ADAPTER_DIR}")

BASELINE_PROMPT = "Explain what a GPTQ quantized model is, in two sentences, to a non-technical reader."

for label, path in [("adapter", ADAPTER_DIR), ("merged fp16", MERGED_DIR)]:
    if path.exists():
        gb = sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1024**3
        print(f"ok   {label:12s} {path}  ({gb:.2f} GB)")
    else:
        print(f"MISS {label:12s} {path}  <- re-run Stage 2")

## Stage 4 — 4-bit GPTQ quantization

GPTQModel — AutoGPTQ is deprecated as of 2026, and GPTQModel is the backend `transformers` now expects for GPTQ checkpoints. Calibration uses 256 `no_robots` samples: the same distribution the model was tuned on, so the quantization error reflects real deployment inputs rather than generic web text.

**Run as a subprocess.** Quantization is the most memory-sensitive stage — it holds the fp16 layers on CPU and moves one at a time to GPU. Any VRAM the notebook kernel is still holding comes straight out of that budget, and `del` + `empty_cache()` never releases the CUDA context itself. Process exit does.

In [ ]:
QUANT_SCRIPT = RUN_DIR / "stage_quantize.py"
QUANT_SCRIPT.write_text(r"""
# Generated by master_pipeline_colab.ipynb — 4-bit GPTQ quantization, run as a
# subprocess so the OS hard-reclaims all VRAM on exit.
import json, sys, time
from pathlib import Path

import torch
from datasets import load_dataset
from gptqmodel import GPTQModel, QuantizeConfig

job = json.loads(Path(sys.argv[1]).read_text())

ds = load_dataset(job["calibration_dataset_id"], split=job["calibration_split"])
ds = ds.shuffle(seed=42).select(range(min(job["num_calibration_samples"], len(ds))))
calibration = [
    "\n".join("{}: {}".format(m["role"], m["content"]) for m in ex["messages"])
    for ex in ds
]
print("calibration samples: {}".format(len(calibration)), flush=True)

qcfg = QuantizeConfig(
    bits=job["bits"],
    group_size=job["group_size"],
    desc_act=job["desc_act"],
    sym=job["sym"],
    damp_percent=job["damp_percent"],
)

t0 = time.time()
model = GPTQModel.load(job["input_dir"], qcfg)
model.quantize(calibration, batch_size=1)
model.save(job["output_dir"])

print("\nGPTQ 4-bit -> {}".format(job["output_dir"]))
print("quantize runtime: {:.1f} min".format((time.time() - t0) / 60))
print("PEAK_VRAM_GB={:.3f}".format(torch.cuda.max_memory_allocated() / 1024**3))
""".lstrip())
print(f"wrote {QUANT_SCRIPT}")

In [ ]:
job = {
    **{k: gptq_cfg[k] for k in (
        "bits", "group_size", "desc_act", "sym", "damp_percent",
        "calibration_dataset_id", "calibration_split", "num_calibration_samples")},
    "input_dir": str(MERGED_DIR),
    "output_dir": str(GPTQ_DIR),
}
job_path = RUN_DIR / "quantize_job.json"
job_path.write_text(json.dumps(job, indent=2))

assert MERGED_DIR.exists(), f"{MERGED_DIR} missing — run Stage 2.5 first"
rc = run_logged([sys.executable, str(QUANT_SCRIPT), str(job_path)], "quantize")
assert rc == 0, f"quantization failed (exit {rc}) — full log at {RUN_DIR / 'quantize.log'}"

In [ ]:
def dir_gb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3 if p.exists() else 0.0

gptq_gb, merged_gb = dir_gb(GPTQ_DIR), dir_gb(MERGED_DIR)
ratio = f"   ({merged_gb / gptq_gb:.1f}x smaller)" if gptq_gb else "   [EMPTY — quantization did not write anything]"
print(f"merged fp16 : {merged_gb:5.2f} GB  {MERGED_DIR}")
print(f"GPTQ 4-bit  : {gptq_gb:5.2f} GB  {GPTQ_DIR}{ratio}")
print(f"free disk   : {shutil.disk_usage(REPO).free / 1024**3:5.2f} GB")

# Stage 5 evaluates SFT as base + peft adapter, so the merged fp16 copy is only
# needed as GPTQ input. Deleting it reclaims ~16 GB — but it's also the thing
# you'd push to the Hub, so this is opt-in rather than automatic.
FREE_MERGED_AFTER_QUANT = False

if FREE_MERGED_AFTER_QUANT:
    assert gptq_gb > 1, "GPTQ output looks empty — refusing to delete the merged model"
    print(f"\ndeleting {MERGED_DIR} ({merged_gb:.1f} GB)")
    shutil.rmtree(MERGED_DIR)
    print(f"free disk now: {shutil.disk_usage(REPO).free / 1024**3:.2f} GB")

## Stage 5 — Evaluate base / SFT / GPTQ

### How the 15 GB constraint shapes this

`eval/run_eval.sh` evaluates at `dtype=bfloat16` — ~16 GB of weights for an 8B model, which does not fit. So **all three checkpoints are evaluated in 4-bit**:

| Checkpoint | How it's loaded | Weights |
|---|---|---|
| base | `google/gemma-4-E4B`, NF4 | ~4.5 GB |
| SFT | the same base in NF4 + `peft=<adapter>` | ~4.5 GB |
| GPTQ | the quantized checkpoint | ~4.5 GB |

Evaluating SFT as *base + adapter* rather than loading the merged fp16 model is what makes it fit, and it costs nothing in fidelity — those are the same weights the merge produced.

**Read the two deltas differently.** `Δ SFT` is NF4 vs NF4, so it isolates instruction tuning cleanly. `Δ Quant` is NF4 vs GPTQ-4bit — a comparison *between two 4-bit schemes*, not fp16 → 4-bit. On a 15 GB card an fp16 reference simply isn't available, so the writeup says it that way rather than implying a full-precision baseline.

Each shot-count group is a separate `lm_eval` invocation in its own subprocess: `--num_fewshot` is global per call, so splitting the run is the only way to honour 5-shot MMLU alongside 25-shot ARC.

In [ ]:
from collections import defaultdict

SHOT_GROUPS = defaultdict(list)
for task in TASKS:
    SHOT_GROUPS[FEWSHOT[task]].append(task)

# A pre-quantized Unsloth repo already carries a quantization_config, so asking
# bitsandbytes to quantize it again would conflict. Only add the bnb args when
# the base is the full-precision canonical repo.
BNB_4BIT = ("" if BASE_IS_PREQUANTIZED else
            f"load_in_4bit=True,bnb_4bit_quant_type=nf4,bnb_4bit_compute_dtype={COMPUTE_DTYPE},")

MODEL_ARGS = {
    "base": f"pretrained={BASE_MODEL},{BNB_4BIT}max_length={MAX_SEQ_LENGTH}",
    "sft":  f"pretrained={BASE_MODEL},peft={ADAPTER_DIR},{BNB_4BIT}max_length={MAX_SEQ_LENGTH}",
    "gptq": f"pretrained={GPTQ_DIR},dtype={COMPUTE_DTYPE},max_length={MAX_SEQ_LENGTH}",
}

def run_eval(label: str, root: Path = RESULTS_DIR) -> None:
    """One lm_eval subprocess per shot-count group; subprocess == guaranteed VRAM reclaim.

    `root` exists so the smoke test can write somewhere the results aggregator in
    §6.2 won't pick up — a 2-example smoke run landing in eval/results/ would
    otherwise shadow the real numbers whenever it happened to be the newer file.
    """
    for shots, tasks in sorted(SHOT_GROUPS.items()):
        out_dir = root / label / f"shots{shots}"
        out_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            sys.executable, "-m", "lm_eval",
            "--model", "hf",
            "--model_args", MODEL_ARGS[label],
            "--tasks", ",".join(tasks),
            "--num_fewshot", str(shots),
            "--batch_size", str(EVAL_BATCH_SIZE),
            "--device", "cuda:0",
            "--output_path", str(out_dir),
        ]
        if EVAL_LIMIT:
            cmd += ["--limit", str(EVAL_LIMIT)]
        print(f"\n{'=' * 72}\n[{label}] {shots}-shot: {', '.join(tasks)}\n{'=' * 72}", flush=True)
        rc = run_logged(cmd, f"eval_{label}_{shots}shot")
        if rc != 0:
            print(f"[{label}] {shots}-shot FAILED (exit {rc}) — continuing with the rest")

for label, args in MODEL_ARGS.items():
    print(f"{label:5s} <- {args}")
print(f"\n{len(SHOT_GROUPS)} lm_eval calls per checkpoint x 3 = {len(SHOT_GROUPS) * 3} runs")

### 5.1 · Smoke test first

Two examples per task. Catches a malformed `model_args`, a missing IFEval extra, or an OOM in about a minute — rather than an hour into the real run.

In [ ]:
_saved_limit, EVAL_LIMIT = EVAL_LIMIT, 2
try:
    run_eval("base", root=RUN_DIR / "smoke")   # kept out of eval/results/ on purpose
finally:
    EVAL_LIMIT = _saved_limit
print(f"\nsmoke test done — EVAL_LIMIT restored to {EVAL_LIMIT}")

### 5.2 · Full runs

Each cell is independent and re-runnable. On a T4 in `quick` mode budget ~30–50 min per checkpoint; in `full` mode each of these is several hours.

In [ ]:
run_eval("base")

In [ ]:
run_eval("sft")

In [ ]:
run_eval("gptq")

## Stage 6 — Results

### 6.1 · Deployment cost: VRAM and throughput

The accuracy table is only half the story — the reason to quantize at all is the other half. Measured in a subprocess per checkpoint so each gets a clean CUDA context and an honest peak-VRAM reading.

In [ ]:
BENCH_SCRIPT = RUN_DIR / "stage_bench.py"
BENCH_SCRIPT.write_text(r"""
# Generated by master_pipeline_colab.ipynb — measures deployment cost (peak VRAM
# and decode throughput) for one checkpoint, in its own CUDA context.
import json, sys, time
from pathlib import Path

import torch
from transformers import AutoTokenizer, BitsAndBytesConfig

job = json.loads(Path(sys.argv[1]).read_text())

kwargs = {"device_map": "cuda:0"}
if job.get("load_in_4bit"):
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=getattr(torch, job["compute_dtype"]),
    )
else:
    kwargs["torch_dtype"] = getattr(torch, job["compute_dtype"])


def load_model(path):
    # Gemma 4 is multimodal; depending on the transformers version it may register
    # under the image-text-to-text auto class rather than causal-LM.
    from transformers import AutoModelForCausalLM
    errors = []
    try:
        return AutoModelForCausalLM.from_pretrained(path, **kwargs)
    except Exception as exc:
        errors.append(exc)
    try:
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(path, **kwargs)
    except Exception as exc:
        errors.append(exc)
    raise RuntimeError("could not load {}: {}".format(path, errors))


tok = AutoTokenizer.from_pretrained(job["tokenizer"])
model = load_model(job["pretrained"])
if job.get("peft"):
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, job["peft"])
model.eval()

inputs = tok(job["prompt"], return_tensors="pt").to("cuda")
with torch.inference_mode():
    model.generate(**inputs, max_new_tokens=8, do_sample=False)   # warm up kernels
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=job["max_new_tokens"], do_sample=False)
    torch.cuda.synchronize()
    elapsed = time.time() - t0

n_new = out.shape[1] - inputs.input_ids.shape[1]
payload = {
    "vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
    "tokens_per_s": n_new / elapsed,
}
Path(job["out_json"]).write_text(json.dumps(payload))
print(json.dumps(payload, indent=2))
""".lstrip())
print(f"wrote {BENCH_SCRIPT}")

In [ ]:
# load_in_4bit=False where the checkpoint already carries its own quantization
# config (a pre-quantized Unsloth repo, or the GPTQ output).
QUANTIZE_ON_LOAD = not BASE_IS_PREQUANTIZED

BENCH_JOBS = {
    "base": {"pretrained": BASE_MODEL,    "tokenizer": BASE_MODEL,       "load_in_4bit": QUANTIZE_ON_LOAD},
    "sft":  {"pretrained": BASE_MODEL,    "tokenizer": str(ADAPTER_DIR), "load_in_4bit": QUANTIZE_ON_LOAD,
             "peft": str(ADAPTER_DIR)},
    "gptq": {"pretrained": str(GPTQ_DIR), "tokenizer": str(GPTQ_DIR),    "load_in_4bit": False},
}

deploy = {}
for label, spec in BENCH_JOBS.items():
    out_json = RUN_DIR / f"bench_{label}.json"
    job_file = RUN_DIR / f"bench_{label}_job.json"
    job_file.write_text(json.dumps({
        **spec,
        "compute_dtype": COMPUTE_DTYPE,
        "prompt": BASELINE_PROMPT,
        "max_new_tokens": 128,
        "out_json": str(out_json),
    }))
    print(f"\n--- benchmarking {label} ---", flush=True)
    rc = run_logged([sys.executable, str(BENCH_SCRIPT), str(job_file)], f"bench_{label}")
    deploy[label] = json.loads(out_json.read_text()) if rc == 0 else {}

deploy

### 6.2 · Comparison table

In [ ]:
import pandas as pd

LABELS = ["base", "sft", "gptq"]
METRIC_KEYS = {
    "mmlu":           "acc,none",
    "arc_challenge":  "acc_norm,none",
    "hellaswag":      "acc_norm,none",
    "gsm8k":          "exact_match,strict-match",
    "truthfulqa_mc2": "acc,none",
    "ifeval":         "inst_level_strict_acc,none",
}

def collect(label: str) -> dict:
    """Merge every results*.json under eval/results/<label>/; newest file wins per task."""
    scores = {}
    files = sorted((RESULTS_DIR / label).rglob("results*.json"), key=lambda p: p.stat().st_mtime)
    if not files:
        print(f"[warn] no results for '{label}' — run Stage 5")
    for path in files:
        results = json.loads(path.read_text()).get("results", {})
        for task, key in METRIC_KEYS.items():
            if results.get(task, {}).get(key) is not None:
                scores[task] = results[task][key]
    return scores

df = pd.DataFrame({label: collect(label) for label in LABELS})
assert not df.empty, "no eval results found — run Stage 5 before this cell"

df = df.reindex([t for t in METRIC_KEYS if t in df.index])
df.index = [f"{t} ({FEWSHOT[t]}-shot)" for t in df.index]
BENCH_ROWS = list(df.index)

for label in LABELS:
    if deploy.get(label):
        df.loc["VRAM, inference (GB)", label] = round(deploy[label]["vram_gb"], 2)
        df.loc["Throughput (tok/s)", label]   = round(deploy[label]["tokens_per_s"], 1)

df.columns = ["Base (NF4)", "+ SFT (NF4)", "+ GPTQ 4-bit"]
df.round(4)

In [ ]:
table = df.copy()
table["Δ SFT"]   = table["+ SFT (NF4)"]  - table["Base (NF4)"]
table["Δ Quant"] = table["+ GPTQ 4-bit"] - table["+ SFT (NF4)"]
table.round(4)

In [ ]:
import matplotlib.pyplot as plt   # re-imported here so §6 works on the restart path

ax = df.loc[BENCH_ROWS].plot(kind="bar", figsize=(11, 5), width=0.78)
ax.set_ylabel("score"); ax.set_xlabel("")
ax.set_title("gemma-4-E4B: base -> QLoRA SFT -> GPTQ 4-bit  "
             f"({MODE} mode, " + (f"limit={EVAL_LIMIT}/task)" if EVAL_LIMIT else "full suite)"))
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
(REPO / "reports" / "figures").mkdir(parents=True, exist_ok=True)
plt.savefig(REPO / "reports" / "figures" / "stage_comparison.png", dpi=150)
plt.show()

### 6.3 · Write the writeup

Renders the table plus the run's provenance into `reports/results_summary.md`. Provenance matters more than usual here: a `quick`-mode number is not a leaderboard number, and the file has to say so on its own, without this notebook next to it.

In [ ]:
from datetime import datetime, timezone

vram_lines = "\n".join(f"| {stage} | {peak:.2f} GB |" for stage, peak in VRAM_LOG.items()) \
             or "| _(re-run Stages 1–2 in this session to record these)_ | |"

noise_note = (
    f"- Scores come from a **{EVAL_LIMIT}-example-per-task** subsample and carry real sampling\n"
    "  noise. They are internally comparable across the three checkpoints (identical tasks,\n"
    "  shots and precision) but not comparable to published leaderboard numbers."
    if EVAL_LIMIT else "- Full benchmark suite, no subsampling."
)

summary = f"""# Results — gemma-4-E4B: base -> QLoRA SFT -> GPTQ 4-bit

Generated by `notebooks/master_pipeline_colab.ipynb` on {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}.

## Run provenance

| | |
|---|---|
| Mode | `{MODE}` |
| GPU | {props.name}, {VRAM_GB:.1f} GB |
| Base model | `{BASE_MODEL}` |
| Dataset | `{sft_cfg['dataset_id']}` |
| Max seq length | {MAX_SEQ_LENGTH} |
| Training | {f'{TRAIN_MAX_STEPS} steps' if TRAIN_MAX_STEPS > 0 else f"{sft_cfg['num_train_epochs']} epochs"} |
| Eval limit | {EVAL_LIMIT if EVAL_LIMIT else 'none (full suite)'} examples/task |
| Eval precision | all three checkpoints in 4-bit (see caveat) |

## Comparison

{table.round(4).to_markdown()}

## Reading the deltas

- **Δ SFT** compares NF4 against NF4 — instruction tuning is the only variable, so this
  delta is clean.
- **Δ Quant** compares NF4 against GPTQ-4bit. It is a comparison *between two 4-bit
  schemes*, not fp16 -> 4-bit. An fp16 reference (~16 GB of weights) does not fit the
  {VRAM_GB:.0f} GB card this pipeline targets, so no full-precision baseline exists here.
{noise_note}

## Peak VRAM by stage

| Stage | Peak |
|---|---|
{vram_lines}

## Notes

<!-- Which benchmarks moved, and why? Fill this in by hand. -->
"""

(REPO / "reports" / "results_summary.md").write_text(summary)
print(summary)

### 6.4 · Persist

The adapter (~50 MB) and the results JSON are the artifacts worth keeping. The merged fp16 model can always be rebuilt from base + adapter, so there's rarely a reason to upload 16 GB of it.

In [ ]:
if DRIVE_DIR:
    for src in (ADAPTER_DIR, RESULTS_DIR, REPO / "reports"):
        shutil.copytree(src, DRIVE_DIR / src.name, dirs_exist_ok=True)
    print(f"copied adapter + results + reports -> {DRIVE_DIR}")

# Push to the Hub (uncomment and set your username):
# HF_USER = "your-username"
# !python scripts/push_to_hub.py {ADAPTER_DIR} {HF_USER}/gemma-4-e4b-no_robots-sft
# !python scripts/push_to_hub.py {GPTQ_DIR}    {HF_USER}/gemma-4-e4b-no_robots-sft-gptq4bit

## Appendix — troubleshooting on a 15 GB card

| Symptom | Cause | Fix |
|---|---|---|
| `Some modules are dispatched on the CPU or the disk` | `device_map` offloaded part of the model; bitsandbytes 4-bit rejects that | check §2 didn't warn that your Unsloth build ignores `text_only`; drop `MAX_SEQ_LENGTH` to 1024 |
| `RuntimeError: ... bf16 is not supported` | T4 is sm_75 | already handled — verify §0.1 printed `bf16 support: False` and §2.3 set `fp16=True` |
| `OutOfMemoryError` loading the base model | E4B under the T4 fp32 fallback needs ~16–20 GB | set `MODEL_TIER = "E2B"` in §0.5 (auto already does this on a T4), or move to an L4/A100 where bf16 avoids the fallback |
| `Using float16 precision for gemma4 won't work! Using float32` | not an error — Gemma's activations exceed fp16's 65504 ceiling, so Unsloth upcasts | expected on any pre-Ampere card; it is the reason E2B is auto-selected. Nothing to fix |
| OOM during training | fp32 activations scale with sequence length | lower `MAX_SEQ_LENGTH`, or `TRAIN_BATCH_SIZE = 1` with a matching `GRAD_ACCUM` in §0.5 |
| Hundreds of `UNEXPECTED` keys in the load report for `vision_tower` / `audio_tower` | `text_only=True` skipped the multimodal towers, so their checkpoint keys go unused | expected and harmless — it's the confirmation `text_only` took effect |
| `TypeError: 'NoneType' object is not iterable` in `unsloth_base_fast_generate` | `text_only=True` leaves `config.architectures = None`, and Unsloth iterates it to detect a VLM | handled in §2 by setting `config.architectures` after load; it also has to be set before saving, or the merged/GPTQ checkpoints reload badly |
| Loss in the hundreds instead of 10–15 | Gemma 4 gradient-accumulation bug | upgrade to `unsloth >= 2026.4.4`; this config uses grad accum, so it is exposed to it |
| `ValueError: Target modules .*language_model\.layers… not found` | `text_only=True` loads `Gemma4ForCausalLM`, whose paths are `model.layers.N.…` — no `language_model.` segment. `configs/lora.yaml` assumes the multimodal wrapper | handled in §2.2, which derives the prefix from the live module tree. `src/slm_prod/train_sft.py` still reads the config regex and will hit this |
| Load report names an `unsloth/*` repo when you passed `google/*` | expected — `FastLanguageModel` resolves known names to its own pre-quantized uploads when `load_in_4bit=True` | nothing to fix; Unsloth works with any transformers model, it just prefers its own fixed 4-bit builds |
| OOM during eval | `--batch_size auto` overshot | `EVAL_BATCH_SIZE = 1` in §0.5 |
| `ImportError: create_recurrent_attention_mask` on `import gptqmodel` | gptqmodel ≥7.3.2 eagerly imports every bundled model definition | the `gptqmodel<7.3.2` pin in `requirements-colab.txt` covers this — check it wasn't upgraded |
| `ImportError: gptqmodel requires optimum version 1.24.0 or higher` during `get_peft_model` | PEFT probes `is_gptqmodel_available()` while attaching adapters; having gptqmodel installed for Stage 4 makes that probe raise on Colab's older optimum | `optimum>=1.24.0` is now in `requirements-colab.txt`; §0.4 checks it before you spend a session finding out |
| IFEval fails on missing `langdetect` / `nltk` | plain `lm-eval` omits the extras | §0.3 installs `lm-eval[ifeval]`; re-run it |
| Merge killed with no traceback | Colab's OOM-killer — you're on the 16 GB `merge_and_unload()` path | use §2.5's `save_pretrained_merged`, not `src/slm_prod/merge_lora.py` |
| `No space left on device` | ~40 GB of artifacts | `FREE_MERGED_AFTER_QUANT = True` in Stage 4 |
| Session died mid-run | free-tier disconnect | `USE_DRIVE = True` in §0.4, then use the Resume cell |
| A subprocess stage fails but prints no traceback | Colab does not reliably forward a subprocess's inherited stdout into cell output | all stages go through `run_logged`, which pipes output into the cell and tees it to `.master_run/<stage>.log` — read that file |
| `Cannot use chat template functions because tokenizer.chat_template is not set` | Gemma 4 keeps its template in a standalone `chat_template.jinja`; `AutoTokenizer` reads only `tokenizer_config.json` ([transformers#45205](https://github.com/huggingface/transformers/issues/45205)) | Stage 1 pulls the `.jinja` with `hf_hub_download` — this is handled, but any code of your own calling `apply_chat_template` on a fresh Gemma 4 tokenizer needs the same treatment |
| `AssertionError: no chat template found ...` in Stage 1 | every source in the chain came up empty | the `-it` mirror is gated *separately* from the base repo — accept its license on your HF account |
| Stage 1 still raises the *old* `assert tokenizer_probe.chat_template` message | Colab is running a cached copy of the notebook, not the current one | File → Revert to saved (or re-open from GitHub); confirm the cell body matches this repo before re-running |
| `DownloadStallError` on `unsloth/gemma-4-E4B-unsloth-bnb-4bit` | low-traffic repo; Xet and HTTP fallback both exhausted (already seen once on Colab — see `configs/model.yaml`) | set `PREFER_UNSLOTH_4BIT = False` in §0.5 to use the canonical `google/` repo, and budget ~11 GB more disk |
| `ValueError: ... already quantized` / conflicting `quantization_config` | passed `load_in_4bit` on top of a pre-quantized repo | handled by `BASE_IS_PREQUANTIZED` in §0.5 — check it matches the base you actually selected |

### Known divergences from `src/slm_prod/`

This notebook deliberately does **not** call two of the repo's entry points, because they don't fit 15 GB as written:

- `src/slm_prod/train_sft.py:69` hardcodes `bf16=True` — raises on any pre-Ampere GPU.
- `src/slm_prod/merge_lora.py:20` loads the 8B base at bf16 (~16 GB) — over both VRAM and Colab RAM.

`eval/run_eval.sh` is bypassed too: it evaluates at `dtype=bfloat16` (doesn't fit) and passes no `--num_fewshot`, so the README's 5/25/10-shot counts were never applied.